In [0]:
%sql
-- 4.2 MEETING METRIC — DAYS_SINCE_LAST_MEETING (CRM signal)
-- Spec: latest CRM ACTIVITY_AT whose text contains 'A new event has been
-- scheduled.' (case-insensitive ILIKE). The same automated Calendly message is
-- EXCLUDED from email recency (3_05) but INTENTIONALLY INCLUDED here — spec
-- 5.1: the email and CRM-meeting uses of the scheduling notification are
-- deliberately different. Verified asymmetry (Nov 12 2026): the 2,366 leads
-- whose only inbox emails are scheduling notifications have no
-- days_since_last_email row but DO have a meeting recency row here.
-- FIX v2 (Nov 12 2026): source moved from lead_activites_email_body to the
-- PARENT body_text_quoted (CAST AS STRING ILIKE). The fragment-extracted child
-- body_text misses the phrase on 102 of 20,308 matching activities — it sits in
-- quoted reply history / a second body fragment the child write dropped (68 of
-- the 102 lost a fragment row) — leaving 21 lead values wrong (5 coaching
-- clients). The parent cast is the full text of record, fragment-complete, and
-- makes 3_05's exclusion predicate IDENTICAL to this table's inclusion
-- predicate (the spec's deliberate same-text mirror). The phrase never appears
-- in SMS texts (verified). No type/status filter per spec: population = 20,093
-- Email/inbox (Calendly to DEA) + 37 Email/sent (staff forwards).
-- Dedup: one current version per ACTIVITY_ID (DATE_UPDATED then INSERT_DATE),
-- then latest qualifying activity per LEAD_ID; recency = DATEDIFF(CURRENT_DATE,
-- DATE(activity_at)). Verified Nov 12 2026: 7,780 leads, 1 row per lead_id,
-- days 0-140, recompute parity exact (0 mismatches), 841/1,381 coaching
-- clients covered (525 within the 30-day flag window). HARDENING Nov 12 2026:
-- deterministic tie-break (activity_id DESC) added to the latest-per-lead
-- rank — no value change (verified).
CREATE OR REPLACE TABLE crm_ingestion.gold.days_since_last_meeting AS (
WITH deduped_id_activities_filtered_text AS (
    SELECT
        id AS activity_id,
        lead_id,
        activity_at,
        ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC, bronze_insert_date DESC) AS id_rn
    FROM
        crm_ingestion.silver.lead_activites_processed
    WHERE
        CAST(body_text_quoted AS STRING) ILIKE '%A new event has been scheduled.%'
),
latest_text_per_lead_id AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY lead_id ORDER BY activity_at DESC, activity_id DESC) AS lead_id_rn
    FROM
        deduped_id_activities_filtered_text
    WHERE
        id_rn = 1
)
SELECT
    lead_id,
    DATEDIFF(CURRENT_DATE, DATE(activity_at)) AS days_since_last_meeting
FROM
    latest_text_per_lead_id
WHERE
    lead_id_rn = 1  
)